# Les Emeutes

## Descrition
On analyse souvent les émeutes comme une explosion soudaine et imprévisible. Est ce vraiment le cas ? Il est possible que les territoires avec un taux d'inactivité plus élevé subissent plus frequemment ces désagréments. L'objectif de ce projet Python est ainsi d'explorer les dynamiques des émeutes et d'identifier les principaux drivers qui accompagnent ces évenements.

À travers un schema de collecte, de nettoyage et d'analyse de données, nous étudirons les corrélations entre présence de quartiers prioritaires (QPV), le taux d'inactivité et les émeutes. 

La difficulté première du projet étant d'obtenir des renseignements fiables sur la présence d'emeutes en France, nous avons choisi de croiser deux sources de données complémentaires: GDELT, qui capture l'activité médiatique et événementielle mondiale en temps réel, et Google Trends, qui reflète l'attention directe des citoyens à travers leurs requêtes de recherche.

## Problématique

## Plan:

- Récupération des données:
    - GDELT
    - Google Trends
- Nettoyage
- Comparaison Gdelt, Google Trends
- Les données sociales:
    - Taux d'Inactivité
    - Quartier prioritaires
- Peut-on trouver une relation ?
- Conclusion


## Récupération des données

Deux bases de données sont utilisées pour caractériser au mieux ce phénomène: GDELT et Google Trends. Avant de les récupérer, importons les librairies



In [ ]:
import google-cloud-bigquery db-dtypes ## Non natif, pour ...
import json
import s3fs

from GT_data.Google_trends import Google_Trends # Créée par nous même
from trendspy import Trends ## Non natif, pour scraper les trends

import pandas as pd
import matplotlib.pyplot as plt

-> GDELT:

Descritption: GDELT est une base de données ouverte qui scanne en continu les médias du monde entier (presse en ligne, agences, web) et transforme chaque article en événement structuré. 

Objectif: La parution d'articles sur le sujet des émeutes est le média le plus précis afin de déterminer où et quand se sont déroulés ces évenements.

Nous prendrons deux sous-bases de GDELT: une petite, précise mais avec peu de lignes; et une grosse necessitant un nettoyage

In [5]:
# --------- GDELT ---------------------

# -- Settings --
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})
MY_BUCKET = "vgounot"

# -- Import --
FILE_PATH_S3 = f"{MY_BUCKET}/diffusion/gcloud_emeutes_all.json"
with fs.open(FILE_PATH_S3, 'r', encoding="utf-8") as file_in:
    df_GDELT_small = pd.read_json(file_in)
    print("GDELT bigQuery small")


FILE_PATH_S3 = f"{MY_BUCKET}/diffusion/2023_gcloud_emeutes.parquet"

with fs.open(FILE_PATH_S3, 'rb', encoding="utf-8") as file_in:
    df_GDELT_large = pd.read_parquet(file_in)
    print("GDELT bigQuery gros")





 GDELT bigQuery small
                                  id  \
0  conflict_2f281a8e827f0d70809bfac9   
1  conflict_e9cf41218a7a2e9b153e9ab5   
2                  conflict_f6108745   
3          conflict_4edb9e28f4e36b78   
4          conflict_87de69f2173d412a   

                                                 url  \
0  https://gdeltcloud.com/events/police-use-riot-...   
1  https://gdeltcloud.com/events/france-mob-attac...   
2  https://gdeltcloud.com/events/bayern-munich-vs...   
3  https://gdeltcloud.com/events/pro-palestine-mi...   
4  https://gdeltcloud.com/events/barcelona-mob-at...   

                                   primary_story_url    family  \
0  https://gdeltcloud.com/stories/police-use-riot...  conflict   
1  https://gdeltcloud.com/stories/france-mob-atta...  conflict   
2  https://gdeltcloud.com/stories/bayern-munich-v...  conflict   
3  https://gdeltcloud.com/stories/pro-palestine-m...  conflict   
4  https://gdeltcloud.com/stories/raging-mob-atta...  conflict   




-> Google Trends

Description: Google Trends permet de connaitre le nombre de requêtes pour un mot clef.
Objectif: avoir un aperçu temporel du ressenti d'émeutes en France. 
Dans un but de vérifier les informations récupérées avec GDELT, Google Trends sert à caractériser l'intérêt des francais pour les emeutes à travers des recherches comme "Mortier", "LBD", "casseurs", ... 



In [6]:
G_trend = Google_Trends()

# Mots de Recherche
mots_contexte = ["Emeute", "violences urbaines"]
mots_acteurs = ["casseurs"]
mots_actions = ["saccage", 'pillage']
mots_matériel = ["mortier", "LBD"]
mots_de_recherche = mots_contexte+mots_acteurs + mots_actions + mots_matériel

# Récupération des données, normalisation et création d'une moyenne de recherche
df = G_trend.get_trends(mots_de_recherche, geo="FR", timeframe="today 3-m")
df = G_trend.normalisation(df)
df = G_trend.add_average(df)

df_google_trends = df[["mean", "Time"]]

G_trend.plot(df_google_trends)

HTTPError: 429 Client Error: Too Many Requests for url: https://www.google.com/sorry/index?continue=https://trends.google.com/trends/embed/explore/TIMESERIES%3Freq%3D%257B%2522comparisonItem%2522%253A%2B%255B%257B%2522keyword%2522%253A%2B%2522Emeute%2522%252C%2B%2522time%2522%253A%2B%2522today%2B3-m%2522%252C%2B%2522geo%2522%253A%2B%2522FR%2522%257D%252C%2B%257B%2522keyword%2522%253A%2B%2522violences%2Burbaines%2522%252C%2B%2522time%2522%253A%2B%2522today%2B3-m%2522%252C%2B%2522geo%2522%253A%2B%2522FR%2522%257D%252C%2B%257B%2522keyword%2522%253A%2B%2522casseurs%2522%252C%2B%2522time%2522%253A%2B%2522today%2B3-m%2522%252C%2B%2522geo%2522%253A%2B%2522FR%2522%257D%252C%2B%257B%2522keyword%2522%253A%2B%2522saccage%2522%252C%2B%2522time%2522%253A%2B%2522today%2B3-m%2522%252C%2B%2522geo%2522%253A%2B%2522FR%2522%257D%255D%252C%2B%2522category%2522%253A%2B0%252C%2B%2522property%2522%253A%2B%2522%2522%257D%26hl%3Den%26tz%3D360&hl=en&q=EgSftO-mGLCFh9gGIi9cIjG-LvLoliiAEMmXosaEb9E3mpUDZkJ6KFhTeQLi

## Comparer GDELT et Google Trends

Maintenant que nous avons les deux bases d'émeutes, nous pouvons mettre en perspective les résultats trouvés. Les évenements trouvés sur GDELT étant discrets, nous allons aggréger les résultats par groupe de 3 jours

In [ ]:
def comparer_GDELT_Gtrend(self, df_gdelt, aggregation_periode = "3D", afficher=False):
    df_gdelt['observed_at'] = pd.to_datetime(df_gdelt['observed_at']).dt.tz_localize(None)

    # Pour aggréger par x jours ( mettre "xD" en periode ou 'W-SUN' pour une semaine)
    df_gdelt_weekly = (
        df_gdelt
        .resample(aggregation_periode, on='observed_at')
        .size()
        .reset_index(name='nb_emeutes')
    )

    # Renommer la colonne pour qu'elle ait le même nom que dans df_trends
    df_gdelt_weekly = df_gdelt_weekly.rename(columns={'observed_at': 'Time'})

    G_trend.comparer_GDELT_plot(df_gdelt_weekly)

G_trend.comparer_GDELT_Gtrend(df_GDELT_small, "3D")

Pour comparer à la table entière:

In [ ]:
G_trend.comparer_GDELT_Gtrend(df_GDELT_large, "3D")